# SCRUM-318: Compare Arabic embedding models

Round 2: BGE-M3 (the one to beat) vs harrier-0.6b, arctic-embed-l-v2.0, bge-m3-law and bahith-4b, plus a BM25 keyword baseline, on Egyptian Labor Law articles. Each model is wrapped in try/except, so one failure will not stop the run.

**Before running:** (1) add your dataset containing `labor_law.jsonl` and `questions.jsonl`, (2) Settings -> Accelerator: GPU, (3) Settings -> Internet: On.

In [8]:
!pip install -q -U sentence-transformers rank_bm25 transformers

In [9]:
import glob, json, re, time, gc
import numpy as np
import pandas as pd
import torch

MAX_LEN = 512  # same truncation for every model, so the comparison is fair

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found. Add your dataset to this notebook."
    return hits[0]

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

DIAC = re.compile(r"[\u064B-\u0652\u0670\u0640]")  # tashkeel + tatweel

def normalize(text):
    # identical to parse_law.py: apply the same function to passages AND queries
    text = DIAC.sub("", text)
    text = re.sub("[إأآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    return re.sub(r"\s+", " ", text).strip()

articles = read_jsonl(find("labor_law.jsonl"))
questions = read_jsonl(find("questions.jsonl"))

doc_ids = [a["id"] for a in articles]
id_to_idx = {d: i for i, d in enumerate(doc_ids)}
bad = [q for q in questions if q["article_number"] not in id_to_idx]
print(len(articles), "articles;", len(questions), "questions;", len(bad), "questions with unknown article")
assert not bad, bad[:3]

passages = [normalize(a["text"]) for a in articles]
queries = [normalize(q["question"]) for q in questions]
gold = np.array([id_to_idx[q["article_number"]] for q in questions])
styles = [q["style"] for q in questions]

208 articles; 80 questions; 0 questions with unknown article


In [10]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def time_queries(fn, texts, n=30):
    # latency of encoding ONE query at a time (what the API would do)
    fn(texts[0]); sync()  # warm-up
    ts = []
    for t in texts[:n]:
        start = time.perf_counter(); fn(t); sync()
        ts.append((time.perf_counter() - start) * 1000)
    return float(np.mean(ts)), float(np.percentile(ts, 95))

def gold_ranks(scores, gold):
    ranks = []
    for i, g in enumerate(gold):
        order = np.argsort(-scores[i])
        ranks.append(int(np.where(order == g)[0][0]) + 1)
    return np.array(ranks)

def metrics(ranks):
    return {
        "recall@1": float(np.mean(ranks <= 1)),
        "recall@3": float(np.mean(ranks <= 3)),
        "recall@5": float(np.mean(ranks <= 5)),
        "MRR@10": float(np.mean(np.where(ranks <= 10, 1.0 / ranks, 0.0))),
    }

def by_style(ranks, styles):
    df = pd.DataFrame({"rank": ranks, "style": styles})
    return {s: metrics(d["rank"].values) for s, d in df.groupby("style")}

In [11]:
from sentence_transformers import SentenceTransformer

CANDIDATES = {
    "bge-m3": dict(name="BAAI/bge-m3"),
    "harrier-0.6b": dict(name="microsoft/harrier-oss-v1-0.6b", prompt_name="web_search_query"),
    "harrier-0.6b + legal instruction": dict(
        name="microsoft/harrier-oss-v1-0.6b",
        q_prefix="Instruct: Given a question about Egyptian labor law, retrieve the article that answers it\nQuery: "),
    "arctic-embed-l-v2.0": dict(name="Snowflake/snowflake-arctic-embed-l-v2.0", q_prefix="query: ", trust_remote_code=True),
    "bge-m3-law": dict(name="mhaseeb1604/bge-m3-law"),
    "bahith-4b (cut to 1024)": dict(name="mabdulaziz499/bahith-arabic-retrieval-4b", truncate_dim=1024,
                                     prompt_name="query", batch=4),   # biggest model, runs last
    # To re-test the old runner-up, uncomment:
    # "multilingual-e5-large": dict(name="intfloat/multilingual-e5-large", q_prefix="query: ", p_prefix="passage: "),
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def l2norm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-9, None)

def load(cfg):
    kw = {"torch_dtype": torch.float16} if device == "cuda" else {}
    extra = {"truncate_dim": cfg["truncate_dim"]} if cfg.get("truncate_dim") else {}
    return SentenceTransformer(cfg["name"], device=device, model_kwargs=kw,
                               trust_remote_code=cfg.get("trust_remote_code", False), **extra)

rows, style_rows, ranks_by_model, top1_by_model, failures = [], [], {}, {}, []

for label, cfg in CANDIDATES.items():
    print("==", label)
    model = None
    try:
        model = load(cfg)
        model.max_seq_length = MAX_LEN

        pn = cfg.get("prompt_name")
        if pn and pn not in (getattr(model, "prompts", None) or {}):
            print(f"  note: prompt '{pn}' is not defined in this model; encoding queries without it")
            pn = None

        def enc_q(texts):
            texts = [cfg.get("q_prefix", "") + t for t in texts]
            return l2norm(model.encode(texts, prompt_name=pn, batch_size=cfg.get("batch", 32),
                                       show_progress_bar=False))

        start = time.perf_counter()
        D = l2norm(model.encode([cfg.get("p_prefix", "") + p for p in passages],
                                batch_size=cfg.get("batch", 16), show_progress_bar=False))
        sync()
        index_s = time.perf_counter() - start

        Q = enc_q(queries)
        dim = D.shape[1]
        if dim != 1024:
            print(f"  WARNING: dim is {dim}, not 1024 - this model would need a database change")
        scores = Q @ D.T  # cosine similarity, same metric as the pgvector index
        ranks = gold_ranks(scores, gold)
        lat_mean, lat_p95 = time_queries(lambda q: enc_q([q]), queries)

        rows.append({"model": label, "dim": dim, **metrics(ranks),
                     "query_ms_mean": lat_mean, "query_ms_p95": lat_p95, "index_s": index_s})
        for st, m in by_style(ranks, styles).items():
            style_rows.append({"model": label, "style": st, **m})
        ranks_by_model[label] = ranks
        top1_by_model[label] = scores.argmax(axis=1)
        print(f"  done: recall@5={metrics(ranks)['recall@5']:.3f}  MRR@10={metrics(ranks)['MRR@10']:.3f}")
    except Exception as e:
        failures.append((label, repr(e)[:400]))
        print("  FAILED:", repr(e)[:400])
    finally:
        model = None
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

device: cuda
== bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  done: recall@5=0.875  MRR@10=0.812
== harrier-0.6b


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

  done: recall@5=0.938  MRR@10=0.848


In [12]:
from rank_bm25 import BM25Okapi

def tok(text):
    out = []
    for w in re.findall(r"\w+", text):
        if w.startswith("ال") and len(w) > 4:
            w = w[2:]  # crude definite-article stripping
        out.append(w)
    return out

bm25 = BM25Okapi([tok(p) for p in passages])
scores = np.array([bm25.get_scores(tok(q)) for q in queries])
ranks = gold_ranks(scores, gold)
lat_mean, lat_p95 = time_queries(lambda q: bm25.get_scores(tok(q)), queries)

rows.append({"model": "bm25 (keywords)", "dim": None, **metrics(ranks),
             "query_ms_mean": lat_mean, "query_ms_p95": lat_p95, "index_s": 0.0})
for s, m in by_style(ranks, styles).items():
    style_rows.append({"model": "bm25 (keywords)", "style": s, **m})
ranks_by_model["bm25 (keywords)"] = ranks
top1_by_model["bm25 (keywords)"] = scores.argmax(axis=1)

In [13]:
results = pd.DataFrame(rows)
results["hits@5"] = [int((ranks_by_model[m] <= 5).sum()) for m in results["model"]]
ref = "bge-m3"
results["hits@5 vs bge-m3"] = results["hits@5"] - int((ranks_by_model[ref] <= 5).sum())
results = results.sort_values("MRR@10", ascending=False).round(3)
by_style_df = pd.DataFrame(style_rows).round(3)

print("OVERALL (", len(questions), "questions ). 'hits@5 vs bge-m3' is the difference in number of questions answered in the top 5.")
display(results)
print("BY QUESTION STYLE")
display(by_style_df.sort_values(["style", "MRR@10"], ascending=[True, False]))
print("Question counts per style:", pd.Series(styles).value_counts().to_dict())
if failures:
    print("\nFAILED MODELS (not in the tables above):")
    for name, err in failures:
        print("-", name, ":", err)

results.to_csv("/kaggle/working/results_overall.csv", index=False)
by_style_df.to_csv("/kaggle/working/results_by_style.csv", index=False)

OVERALL ( 80 questions ). 'hits@5 vs bge-m3' is the difference in number of questions answered in the top 5.


,model,dim,recall@1,recall@3,recall@5,MRR@10,query_ms_mean,query_ms_p95,index_s,hits@5,hits@5 vs bge-m3
5,bahith-4b (cut to 1024),1024.0,0.762,0.925,0.938,0.848,70.202,78.721,14.151,75,5
3,arctic-embed-l-v2.0,1024.0,0.788,0.888,0.912,0.840,22.236,30.123,1.078,73,3
0,bge-m3,1024.0,0.750,0.850,0.875,0.812,23.416,30.125,1.057,70,0
4,bge-m3-law,1024.0,0.638,0.838,0.862,0.745,22.540,29.865,1.095,69,-1
1,harrier-0.6b,1024.0,0.638,0.738,0.800,0.704,54.011,57.550,2.883,64,-6
2,harrier-0.6b + legal instruction,1024.0,0.562,0.750,0.750,0.661,54.866,60.878,2.880,60,-10
6,bm25 (keywords),NaN,0.400,0.575,0.650,0.497,0.827,1.187,0.000,52,-18


BY QUESTION STYLE


,model,style,recall@1,recall@3,recall@5,MRR@10
9,arctic-embed-l-v2.0,colloquial,0.842,0.930,0.930,0.883
0,bge-m3,colloquial,0.789,0.877,0.895,0.845
15,bahith-4b (cut to 1024),colloquial,0.772,0.912,0.930,0.845
12,bge-m3-law,colloquial,0.649,0.860,0.895,0.762
3,harrier-0.6b,colloquial,0.684,0.789,0.825,0.747
6,harrier-0.6b + legal instruction,colloquial,0.561,0.737,0.737,0.658
18,bm25 (keywords),colloquial,0.404,0.544,0.632,0.489
1,bge-m3,formal,1.000,1.000,1.000,1.000
7,harrier-0.6b + legal instruction,formal,1.000,1.000,1.000,1.000
16,bahith-4b (cut to 1024),formal,1.000,1.000,1.000,1.000


Question counts per style: {'colloquial': 57, 'scenario': 17, 'formal': 6}


In [14]:
# Do other models fix BGE-M3's misses, or break its hits? (useful for deciding on hybrid search later)
rb = ranks_by_model[ref]
comp = []
for m, rm in ranks_by_model.items():
    if m == ref:
        continue
    comp.append({"model": m,
                 "fixes bge-m3 misses": int(((rb > 5) & (rm <= 5)).sum()),
                 "loses bge-m3 hits": int(((rb <= 5) & (rm > 5)).sum())})
display(pd.DataFrame(comp))

,model,fixes bge-m3 misses,loses bge-m3 hits
0,harrier-0.6b,2,8
1,harrier-0.6b + legal instruction,3,13
2,arctic-embed-l-v2.0,4,1
3,bge-m3-law,1,2
4,bahith-4b (cut to 1024),7,2
5,bm25 (keywords),3,21


In [15]:
# Error analysis: questions the best model got wrong (gold article not in top 5)
best = results.iloc[0]["model"]
print("Best model by MRR@10:", best)
wrong = np.where(ranks_by_model[best] > 5)[0]
for i in wrong:
    got = doc_ids[top1_by_model[best][i]]
    print(f"- gold article {questions[i]['article_number']} (rank {ranks_by_model[best][i]}), top-1 was article {got}")
    print("  ", questions[i]["question"])

Best model by MRR@10: bahith-4b (cut to 1024)
- gold article 35 (rank 45), top-1 was article 57
   هل ينفع صاحب الشركة يقبض راجل أكتر من ست وهما بيعملوا نفس الشغلانة بالظبط؟
- gold article 38 (rank 22), top-1 was article 118
   لو استقلت من الشركة، قانوناً يدوني فلوسي المتأخرة في خلال مهلة أد إيه؟
- gold article 64 (rank 34), top-1 was article 119
   هل ينفع اتجازى من غير ما أكتب أقوالي ويتحقق معايا رسمي وفي خلال كام يوم؟
- gold article 68 (rank 7), top-1 was article 69
   مدير المصنع أصدر قرار مباشر بفصلي من شغلي بسبب تأخيري، هل القرار دا باطل ولازم محكمة هي اللي تفصل؟
- gold article 120 (rank 8), top-1 was article 165
   هل ينفع يفصلوني عشان دخلت النقابة أو اشتكيت صاحب العمل في مكتب التفتيش؟
